# Data collection

In [ ]:
import requests
import time
import pandas as pd
from dotenv import load_dotenv

# Load environment variables from the .env file
load_dotenv()

# Grab the key securely
API_KEY = os.getenv("RIOT_API_KEY")

if not API_KEY:
    print("WARNING: API Key not found. Check your .env file!")

headers = {
    "X-Riot-Token": API_KEY
}

# Riot API uses two different routing systems depending on the endpoint.
# Platform routing (na1, euw1) is used for ladders and summoner profiles.
# Regional routing (americas, europe) is used for match histories.
PLATFORM = "euW1" 
REGION = "europe"

## Ratelimit helper

In [3]:
def make_api_request(url):

    #A wrapper for requests.get() that automatically handles standard Riot API rate limits.
    while True:
        response = requests.get(url, headers=headers)
        
        if response.status_code == 200:
            return response.json()
            
        elif response.status_code == 429: # Too Many Requests 
            retry_after = int(response.headers.get("Retry-After", 10)) 
            print(f"Rate limit hit. Sleeping for {retry_after} seconds...")
            time.sleep(retry_after)
            
        elif response.status_code in [403, 401]:
            print("API Key expired or invalid. Please check your Riot Developer portal.")
            return None
            
        else:
            print(f"Error {response.status_code} for URL: {url}")
            return None

## Challenger ladder

In [12]:
def get_challenger_puuids(limit=10):
    #Pulls the top Ranked Solo Challenger players and retrieves their PUUIDs.

    print("Fetching Challenger Ladder...")
    ladder_url = f"https://{PLATFORM}.api.riotgames.com/lol/league/v4/challengerleagues/by-queue/RANKED_SOLO_5x5"
    ladder_data = make_api_request(ladder_url)
    
    if not ladder_data:
        return []

    # Get the top X players from the ladder entries
    entries = ladder_data.get('entries', [])[:limit]
    
    puuids = []
    print(f"Extracting PUUIDs for the top {limit} players...")
    
    for entry in entries:
        # Riot's newer API data structures use PUUID
        if 'puuid' in entry:
            puuids.append(entry['puuid'])
            
        # Fallback just in case some legacy data still relies on summonerId
        elif 'summonerId' in entry:
            summoner_id = entry['summonerId']
            sum_url = f"https://{PLATFORM}.api.riotgames.com/lol/summoner/v4/summoners/{summoner_id}"
            sum_data = make_api_request(sum_url)
            
            if sum_data and 'puuid' in sum_data:
                puuids.append(sum_data['puuid'])
                
            time.sleep(1.2) 
        
    return puuids

# Test the ladder approach
ladder_seed_puuids = get_challenger_puuids(limit=50)
print(f"Successfully grabbed {len(ladder_seed_puuids)} PUUIDs from the ladder.")

print("Discovered PUUIDs:")
for p in list(ladder_seed_puuids):
    print(p) 

Fetching Challenger Ladder...
Extracting PUUIDs for the top 50 players...
Successfully grabbed 50 PUUIDs from the ladder.
Discovered PUUIDs:
9QU1RJrTmcqoiJUFdKJ_EFqDeJXJj3cNbTKOQt4m1zMwEZMG0-PJSggQQVD6-_Qf8Jif9sCv4NXC7A
QEmbK68FzWs1-H9fKJcNWHjwo-3gvgo4JlSPmP5AgUOTOuTJyEBKG79l9pQ2eYk1Jz7MQZe1RAALFg
SrwO500nVYXb0vPwl6GnY7sjHo26aGFlR25Rlc_0v1g9R69XI-4pDrKWuyDd6ZqkDRFN6T6W37fftg
5I1UMZ8p0-62BaxAT4GOHw7s78OkFtKNhTAZ8PCg2N9gXthOLdnzB5VtBRI59fW4xZhc5k4zD2iwnw
40v4to9yQZO9XesvkPH7v5l0gaO1pUcLiOmiguGSZqfuKca-Ju4uDmlHcpC4kw4U2DNjsarfcwftKw
QEzkyWr_Kzu1GYc9orIHtenNju8s4hDiFxj8SV_sy6dpOpD_CNvEVtWJFtAuDpLjBlXPsHv2iHaAAw
G4vjetNgV9MmjTEcN3gd7etfoKu8lHDUUy3_wMMwTtlgZO_1aJRGMUdFS-mLSVWHv9W1bzBE2tzRDQ
2mFZA1T0-22TH2kGN5PrfFHjCz3PB9KjgZscquETii6xg6pJrEImZVM6jdvfKJxC7zN1bu6ayMGbKw
HF2n2raupaopgVbEoT7zqTBGtC17g9u-kzYccQTQVQGBlLTPzMPVX1LtmKfRu_-1mTB2zmYPMQSk1Q
FI9Hsq9ESjSYnuerHaUrRQ2PTN0FIXv_ZOASyw_XJO1sTex44-CaX3Lybyu2Q29avn4PXlvPkngWAQ
ItYMxkUDnFyk1vPpVKD7ICvfpwz0H0PGKoy3_5QADoaQDQsFPh3RveKgVxkVXtlK0yhk2

## Snowball Method

In [ ]:
def snowball_players_from_seed(seed_puuid, match_count=5):

    #Takes a single player, gets their recent matches, and extracts all other players from those matches.
    # queue=420 restricts the search to Ranked Solo/Duo games
    
    matches_url = f"https://{REGION}.api.riotgames.com/lol/match/v5/matches/by-puuid/{seed_puuid}/ids?queue=420&start=0&count={match_count}"
    match_ids = make_api_request(matches_url)
    
    if not match_ids:
        print("No matches found or error occurred.")
        return set()

    discovered_puuids = set() # Using a set prevents duplicate players
    
    print(f"Found {len(match_ids)} matches. Scraping other players...")
    
    for match_id in match_ids:
        match_url = f"https://{REGION}.api.riotgames.com/lol/match/v5/matches/{match_id}"
        match_data = make_api_request(match_url)
        
        if match_data and 'metadata' in match_data:
            # The metadata block contains a clean list of all 10 participants' PUUIDs
            participants = match_data['metadata']['participants']
            for p in participants:
                discovered_puuids.add(p)
                
        time.sleep(1.2) # Rate limit safety

    # Remove the seed player from the list of newly discovered ones (optional)
    if seed_puuid in discovered_puuids:
        discovered_puuids.remove(seed_puuid)
        
    return discovered_puuids

# Test the snowball approach using the first PUUID we got from the ladder above
if ladder_seed_puuids:
    seed = ladder_seed_puuids[0]
    print(f"Snowballing from seed PUUID: {seed[:15]}...")
    new_players = snowball_players_from_seed(seed, match_count=3)
    print(f"Discovered {len(new_players)} new players to analyze!")

    print("Discovered PUUIDs:")
    for p in list(new_players):
        print(p) 

## Get match history and parse the data

In [9]:
def get_player_match_history(puuid, match_count=20):
    """
    Fetches a list of Match IDs for a specific player, filtering for Ranked Solo/Duo.
    Note: The API limits the 'count' parameter to a maximum of 100 per request.
    """
    print(f"Fetching {match_count} match IDs for player...")
    
    # queue=420 ensures we only look at Ranked Solo/Duo
    url = f"https://{REGION}.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids?queue=420&start=0&count={match_count}"
    
    match_ids = make_api_request(url)
    return match_ids if match_ids else []

In [ ]:
def parse_match_data(match_id, target_puuid):
    """
    Pulls the detailed stats for a match and extracts the specific 
    variables needed for session analysis.
    """
    url = f"https://{REGION}.api.riotgames.com/lol/match/v5/matches/{match_id}"
    match_data = make_api_request(url)
    
    if not match_data or 'info' not in match_data:
        return None

    info = match_data['info']
    
    # Find our target player among the 10 participants
    participants = info.get('participants', [])
    player_data = next((p for p in participants if p['puuid'] == target_puuid), None)
    
    if not player_data:
        return None

    # Extract the core data points for your project
    parsed_data = {
        "match_id": match_id,
        "puuid": target_puuid,
        "game_start_timestamp": info.get("gameStartTimestamp"), # Milliseconds since epoch
        "game_end_timestamp": info.get("gameEndTimestamp"),     # Milliseconds since epoch
        "game_duration_sec": info.get("gameDuration"),          # Seconds
        "win": player_data.get("win"),                          # Boolean (True/False)
        "champion_played": player_data.get("championName"),
        "kills": player_data.get("kills"),
        "deaths": player_data.get("deaths"),
        "assists": player_data.get("assists")
    }
    
    return parsed_data

## Player Rank

In [13]:
def get_player_rank(puuid):
    """
    Fetches a player's Ranked Solo/Duo rank directly using their PUUID.
    Returns a string like 'GOLD II' or 'UNRANKED'.
    """
    league_url = f"https://{PLATFORM}.api.riotgames.com/lol/league/v4/entries/by-puuid/{puuid}"
    league_data = make_api_request(league_url)
    
    if league_data is None:
        return "API_ERROR"
        
    if not league_data: # If the list is empty, they haven't played ranked
        return "UNRANKED"
        
    # A player can have multiple ranks (Solo/Duo, Flex). We only want Solo/Duo.
    for queue in league_data:
        if queue.get('queueType') == 'RANKED_SOLO_5x5':
            tier = queue.get('tier', '')
            rank = queue.get('rank', '')
            return f"{tier} {rank}" # e.g., "DIAMOND IV"
            
    return "UNRANKED (Solo/Duo)"

## Test

In [16]:
if ladder_seed_puuids:
    target_player = ladder_seed_puuids[0]
    
    # 1. Get their last 10 Ranked games
    recent_match_ids = get_player_match_history(target_player, match_count=100)
    
    # 2. Parse the details for each game and store in a list
    all_match_records = []
    
    if recent_match_ids:
        print(f"Parsing details for {len(recent_match_ids)} matches...")
        total_matches = len(recent_match_ids)

        for m_id, index in recent_match_ids, range(total_matches):
            print(f"Processing match {index + 1}/{total_matches} (ID: {m_id})...")
            record = parse_match_data(m_id, target_player)
            if record:
                all_match_records.append(record)
            
            # Gentle sleep to respect rate limits
            time.sleep(1.2)
            
        # 3. Convert to a Pandas DataFrame for easy viewing and analysis
        df = pd.DataFrame(all_match_records)
        
        # Convert the milliseconds timestamps to readable Datetime objects
        df['game_end_datetime'] = pd.to_datetime(df['game_end_timestamp'], unit='ms')
        
        print("\nSuccess! Here is your structured data:")
        display(df.head())

Fetching 100 match IDs for player...
Parsing details for 100 matches...


ValueError: too many values to unpack (expected 2)